# Hyperparameter Sweep — Silver Corpus Size

This notebook explores how the amount of distantly-supervised silver
data affects the joint training strategy (Config 5). We hold the BC5CDR
gold corpus fixed at 5,119 sentences and vary the PubMed silver corpus
in log-2 spaced steps: 5,119 (1:1), 10,238 (2:1), 20,476 (4:1). Together
with the full-silver baseline (40,946 sentences ≈ 8:1) we obtain a
four-point scaling curve.

Three experiments, one notebook. Run them sequentially or pick any
single cell to re-run individually.

**Expected runtime**: ~1.1× the baseline (~3 h total with 3 seeds) on a
single RTX 3060/T4-class GPU; less than the original config 5 because
every variant uses fewer total training sentences.

## 1. Setup — run once per session

In [ ]:
import os
import sys
from pathlib import Path

# Work from the project root so YAML relative paths resolve correctly.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')

import torch, transformers
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Transformers: {transformers.__version__}')

from src.config import load_config
from src.training.trainer_joint import train_joint
from src.utils.aggregate import aggregate_seeds
from src.utils.logging import configure_logging

# One log file per notebook session.
configure_logging(PROJECT_ROOT / 'outputs' / 'logs' / 'sweep_silver', run_name='session')

## 2. Subsample the silver corpus

Builds three subsampled JSONL files alongside the original silver corpus:

* `dataset/pubmed/pubmed_scrapping_1to1.jsonl` — 5,119 sentences
* `dataset/pubmed/pubmed_scrapping_2to1.jsonl` — 10,238 sentences
* `dataset/pubmed/pubmed_scrapping_4to1.jsonl` — 20,476 sentences

Shuffle uses a fixed seed (42) so subsets are reproducible and nested
(`1to1 ⊂ 2to1 ⊂ 4to1`). Skips work if the target files already exist.

In [ ]:
import random

SILVER_SOURCE = Path('dataset/pubmed/pubmed_scrapping.jsonl')
SILVER_SUBSAMPLES = [
    (5119, '1to1'),
    (10238, '2to1'),
    (20476, '4to1'),
]
SHUFFLE_SEED = 42

assert SILVER_SOURCE.is_file(), f'Missing source: {SILVER_SOURCE}'

targets = [SILVER_SOURCE.parent / f'pubmed_scrapping_{suffix}.jsonl' for _, suffix in SILVER_SUBSAMPLES]
if all(t.is_file() for t in targets):
    for t in targets:
        print(f'  {t.name}: already exists ({t.stat().st_size:,} bytes) — skipping')
else:
    print(f'Reading {SILVER_SOURCE}...')
    lines = SILVER_SOURCE.read_text(encoding='utf-8').splitlines()
    print(f'Loaded {len(lines):,} sentences. Shuffling with seed={SHUFFLE_SEED}.')
    rng = random.Random(SHUFFLE_SEED)
    rng.shuffle(lines)
    for size, suffix in SILVER_SUBSAMPLES:
        out = SILVER_SOURCE.parent / f'pubmed_scrapping_{suffix}.jsonl'
        out.write_text('\n'.join(lines[:size]) + '\n', encoding='utf-8')
        print(f'  Wrote {size:,} sentences to {out.name} ({out.stat().st_size:,} bytes)')

## 3. Verify dataset files (originals + subsamples)

In [ ]:
for rel in [
    'dataset/bc5cdr/bc5cdr_train.jsonl',
    'dataset/bc5cdr/bc5cdr_validation.jsonl',
    'dataset/bc5cdr/bc5cdr_test.jsonl',
    'dataset/pubmed/pubmed_scrapping.jsonl',
    'dataset/pubmed/pubmed_scrapping_1to1.jsonl',
    'dataset/pubmed/pubmed_scrapping_2to1.jsonl',
    'dataset/pubmed/pubmed_scrapping_4to1.jsonl',
    'dataset/pubmed/pubmed_test.jsonl',
]:
    p = Path(rel)
    status = 'OK' if p.exists() else 'MISSING'
    size = p.stat().st_size if p.exists() else 0
    print(f"  {rel}: {status}  ({size:,} bytes)")

## 4. Pick seeds and helper

* **Single seed** (`SEEDS = [42]`): fastest, ~1 h total. Treat results
  as directional — small differences between variants may be noise.
* **Three seeds** (`SEEDS = [42, 1337, 2024]`): paper-grade mean ± std,
  ~3 h total. Recommended for final reporting.

Set `SMOKE_TEST = True` for a 1-epoch / 16-sentence pipeline check.

In [ ]:
SEEDS = [42, 1337, 2024]
SMOKE_TEST = False

def run_sweep(config_path: str, seeds=None, smoke_test=None) -> None:
    """Train one sweep variant across the given seeds, then aggregate."""
    seeds = seeds if seeds is not None else SEEDS
    smoke_test = smoke_test if smoke_test is not None else SMOKE_TEST
    cfg = load_config(config_path)
    silver_src = next((s for s in cfg.data.train_sources if s.source_tag == 'silver'), None)
    silver_path = silver_src.path if silver_src else 'n/a'
    print('=' * 70)
    print(f'  {cfg.experiment.name}  |  silver={silver_path}  |  seeds={seeds}')
    print('=' * 70)
    for seed in seeds:
        print(f'\n>>> {cfg.experiment.name} — seed {seed}\n')
        train_joint(config=cfg, seed=seed, smoke_test=smoke_test)
    print(f'\n>>> Aggregating {cfg.experiment.name}')
    aggregate_seeds(cfg.experiment.name, base_dir=str(PROJECT_ROOT / 'outputs'))

print(f'Using SEEDS={SEEDS}, SMOKE_TEST={SMOKE_TEST}')

## 5. Sweep #1 — silver ratio 1:1

5,119 silver sentences (balanced with BC5CDR gold). Smallest dataset,
fastest to train (~0.22× baseline duration).

In [ ]:
run_sweep('configs/sweeps/sweep_silver_1_1.yaml')

## 6. Sweep #2 — silver ratio 2:1

10,238 silver sentences (~0.33× baseline duration).

In [ ]:
run_sweep('configs/sweeps/sweep_silver_2_1.yaml')

## 7. Sweep #3 — silver ratio 4:1

20,476 silver sentences (~0.56× baseline duration).

In [ ]:
run_sweep('configs/sweeps/sweep_silver_4_1.yaml')

## 8. View aggregated results per sweep

Each `run_sweep(...)` call already wrote `aggregated_results.{json,md}`
into `outputs/results/<sweep_name>/`. The cell below renders every
sweep's Markdown table plus the original Config 5 baseline (silver 8:1)
for side-by-side comparison.

In [ ]:
from IPython.display import Markdown, display

configs_to_show = [
    'config_5_joint_uniform',   # baseline, silver 8:1
    'sweep_silver_4_1',
    'sweep_silver_2_1',
    'sweep_silver_1_1',
]
for name in configs_to_show:
    md_file = Path('outputs/results') / name / 'aggregated_results.md'
    if md_file.exists():
        display(Markdown(md_file.read_text(encoding='utf-8')))
    else:
        print(f'[no aggregated_results.md yet] {name}')

## 9. Plot the silver-corpus scaling curve

Reads every available `aggregated_results.json` and plots mean ± std F1
against the silver:gold ratio on a log-2 x-axis. Skips sweeps whose
results have not been generated yet, so the plot still works mid-run.

In [ ]:
import json
import matplotlib.pyplot as plt

POINTS = [
    ('sweep_silver_1_1', 1),
    ('sweep_silver_2_1', 2),
    ('sweep_silver_4_1', 4),
    ('config_5_joint_uniform', 8),
]

def _load_overall_f1(name: str, test_set: str):
    fp = Path('outputs/results') / name / 'aggregated_results.json'
    if not fp.exists():
        return None
    data = json.loads(fp.read_text(encoding='utf-8'))
    block = data.get('test_sets', {}).get(test_set)
    if not block:
        return None
    f1 = block['overall']['f1']
    return float(f1['mean']), float(f1['std'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
for ax, test_set, title in zip(
    axes,
    ['test_bc5cdr', 'test_pubmed'],
    ['BC5CDR test (Chemical + Disease)', 'PubMed test (Virus + Gene)'],
):
    xs, means, stds = [], [], []
    for name, ratio in POINTS:
        stat = _load_overall_f1(name, test_set)
        if stat is None:
            continue
        xs.append(ratio)
        means.append(stat[0])
        stds.append(stat[1])
    if not xs:
        ax.set_title(f'{title}\n(no data yet)')
        continue
    ax.errorbar(xs, means, yerr=stds, marker='o', capsize=4, linewidth=2)
    ax.set_xscale('log', base=2)
    ax.set_xticks([1, 2, 4, 8])
    ax.set_xticklabels(['1:1', '2:1', '4:1', '8:1'])
    ax.set_xlabel('Silver : Gold ratio (log scale)')
    ax.set_ylabel('Overall F1 (mean ± std)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
fig.suptitle('Effect of silver corpus size on joint training (Config 5)')
fig.tight_layout()
plt.show()

## 10. Comparison table — Overall F1 across the four ratios

Compact summary suitable for pasting into a paper or report. Reads the
aggregated JSON files directly so the numbers are always in sync with
the latest run.

In [ ]:
rows = []
for name, ratio in POINTS:
    fp = Path('outputs/results') / name / 'aggregated_results.json'
    if not fp.exists():
        rows.append((ratio, name, '—', '—'))
        continue
    data = json.loads(fp.read_text(encoding='utf-8'))
    summary = {}
    for test_set, label in [('test_bc5cdr', 'BC5CDR'), ('test_pubmed', 'PubMed')]:
        block = data.get('test_sets', {}).get(test_set)
        if not block:
            summary[label] = '—'
            continue
        f1 = block['overall']['f1']
        summary[label] = f"{f1['mean']:.3f} ± {f1['std']:.3f}"
    rows.append((ratio, name, summary.get('BC5CDR', '—'), summary.get('PubMed', '—')))

header = f"| Silver:Gold | Config | BC5CDR F1 | PubMed F1 |"
sep = '|---|---|---|---|'
print(header)
print(sep)
for ratio, name, bc5cdr_f1, pubmed_f1 in rows:
    print(f"| {ratio}:1 | {name} | {bc5cdr_f1} | {pubmed_f1} |")